In [3]:
import xarray
import pandas as pd
import numpy as np
import zipfile
import os

## Process ERA5 Wind Speeds

In [211]:
!unzip ../raw/era5/408ae047535c400a22c47ba04a31425.zip

Archive:  ../raw/era5/408ae047535c400a22c47ba04a31425.zip
  inflating: data.grib               


In [212]:
xgrib = xarray.open_dataset('./data.grib', engine='cfgrib')
df = xgrib.to_dataframe()

/Users/zachl/Desktop/WindEnergyForecasting/venv/lib/python3.12/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [215]:
# This df contains the u and v velocity components at 100 m (u100, v100) at the following 4 lat/long coords:
#   (32.75, -100.80), (32.75, -100.55), (32.50, -100.80), (32.50, -100.55)
# The Pyron windfarm is bounded within these coordinates, and so in order to simplify our dataset
# we take the average of the 4 points to be the actual u100 and v100 wind speeds
# as the Pyron windfarm is approximately in the center of the 4 points

u100df = df.u100
v100df = df.v100

# We assume that the multi-indexed axes/levels are the same across v100 and u100 dataframes
levels = u100df.axes[0].levels
times = levels[0]
lats = levels[1]
longs = levels[2]

averages = pd.DataFrame(columns=['time', 'u100', 'v100', 'speed'])

for time in times:
    # Get all indices with this time, where indices with this time will have the value True, otherwise False
    idxs = u100df.index.get_level_values(0).isin([time])
    # There should be exactly 4 indices for each time, as each time has 4 lat/long coordinates at which speed is measured
    assert idxs.sum() == 4
    # Take the speeds at these indices and average them across lat, then average across long
    u100avg = u100df[idxs].groupby(level=1).mean().mean()
    v100avg = v100df[idxs].groupby(level=1).mean().mean()
    # Take the norm of both components to produce the final speed (>= 0)
    speed = np.sqrt((u100avg ** 2) + (v100avg ** 2))
    new_row = pd.DataFrame({'time': [time], 'u100': [u100avg], 'v100':[v100avg], 'speed':[speed]})
    averages = pd.concat([averages, new_row], ignore_index=True) 
averages.to_csv('./windspeeds_2020-24.csv')
print(averages)

/var/folders/rj/tjcj95tx32q97ft46gzv_wym0000gn/T/ipykernel_32877/2451372666.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  averages = pd.concat([averages, new_row], ignore_index=True)


                     time      u100      v100     speed
0     2020-01-01 00:00:00 -1.832055  5.292307  5.600441
1     2020-01-01 01:00:00 -2.570372  6.719890  7.194702
2     2020-01-01 02:00:00 -2.180511  7.193185  7.516418
3     2020-01-01 03:00:00 -1.567451  7.101928  7.272845
4     2020-01-01 04:00:00 -0.809609  7.147148  7.192857
...                   ...       ...       ...       ...
43843 2024-12-31 19:00:00 -2.095122 -3.899143  4.426382
43844 2024-12-31 20:00:00 -2.087258 -3.301058  3.905590
43845 2024-12-31 21:00:00 -2.272206 -3.115777  3.856292
43846 2024-12-31 22:00:00 -2.603068 -2.155322  3.379553
43847 2024-12-31 23:00:00 -2.881468 -1.930670  3.468478

[43848 rows x 4 columns]


## Process Power Generated

In [155]:
def unzip(file, destination):
    try:
        with zipfile.ZipFile(file, 'r') as zip_ref:
            zip_ref.extractall(destination)
    except zipfile.BadZipFile:
        print(f"Error: '{file}' is not a valid ZIP file.")
    except FileNotFoundError:
        print(f"Error: The file '{file}' was not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
import os

years = ['2020', '2021', '2022', '2023', '2024']
process_dir = '../raw/power/to_process'
for year in years:
    directory = "../raw/power/" + str(year)
    unzipped = os.path.join(directory, 'unzipped')
    # Unzip all month directories
    for monthfile in os.listdir(directory):
        path = os.path.join(directory, monthfile)
        if os.path.isfile(path):
            if(monthfile == '.DS_Store'): continue
            month = monthfile[-6:-4]
            destination = os.path.join(unzipped, month)
            unzip(path, destination) 

    # Go through each month directory and get .csvs to process
    for monthfile in os.listdir(unzipped):
        monthpath = os.path.join(unzipped, monthfile)
        if(os.path.isdir(monthpath)):
            days = {}
            max_day = 0
            for timefile in os.listdir(monthpath):
                if(timefile == '.DS_Store'): continue
                hour = timefile[49:51]
                if hour[0] == '.':
                    hour = timefile[50:52]
                # We only care about the 23rd hour because the corresponding .csv file for this hour will contain 
                # the power generated for all hours in the last few days
                if hour == '23':
                    zip_file_path = os.path.join(monthpath, timefile)
                    destination = os.path.join(process_dir)
                    unzip(zip_file_path, destination)

In [240]:
def toHourString(time):
    if time < 10:
        return ' 0' + str(time) + ':00:00'
    else:
        return ' ' + str(time) + ':00:00'

def reformatDate(date):
    month = date[0:2]
    day = date[3:5]
    year = date[6:]
    return year + '-' + month + '-' + day

In [241]:
# Go through .csvs to process and add to dataframe
power_df = pd.DataFrame(columns=[])
num_files = len(os.listdir(process_dir))
i = 1
for file in os.listdir(process_dir):
    print(f"{(i / num_files) * 100}% : Processing file {i} of {num_files}")
    df = pd.read_csv(os.path.join(process_dir, file))
    # For some reason, some of the .csvs use the column GEN_LZ_WEST instead of ACTUAL_LZ_WEST
    df = df.rename(columns={'GEN_LZ_WEST': 'ACTUAL_LZ_WEST'})
    # Columns to keep
    columns_to_keep = ['DELIVERY_DATE', 'HOUR_ENDING', 'ACTUAL_LZ_WEST']
    # Drop all columns except the ones specified
    df = df.drop(df.columns.difference(columns_to_keep), axis=1)
    power_df = pd.concat([power_df, df], ignore_index=True)
    i += 1
print('Dropping NaNs and duplicates:')
power_df = power_df.dropna()
power_df = power_df.drop_duplicates(subset=['DELIVERY_DATE', 'HOUR_ENDING'])
power_df['datetime'] = power_df['DELIVERY_DATE'].map(reformatDate) + power_df['HOUR_ENDING'].map(toHourString)
power_df = power_df.drop('DELIVERY_DATE', axis=1)
power_df = power_df.drop('HOUR_ENDING', axis=1)
power_df = power_df.sort_values(by='datetime')
print('Finished processing!')
power_df.to_csv('./power_2020-24.csv')
print(power_df)

0.05473453749315819% : Processing file 1 of 1827
0.10946907498631638% : Processing file 2 of 1827
0.16420361247947454% : Processing file 3 of 1827
0.21893814997263275% : Processing file 4 of 1827
0.27367268746579093% : Processing file 5 of 1827
0.3284072249589491% : Processing file 6 of 1827
0.38314176245210724% : Processing file 7 of 1827
0.4378762999452655% : Processing file 8 of 1827
0.49261083743842365% : Processing file 9 of 1827
0.5473453749315819% : Processing file 10 of 1827
0.60207991242474% : Processing file 11 of 1827
0.6568144499178982% : Processing file 12 of 1827
0.7115489874110563% : Processing file 13 of 1827
0.7662835249042145% : Processing file 14 of 1827
0.8210180623973727% : Processing file 15 of 1827
0.875752599890531% : Processing file 16 of 1827
0.9304871373836892% : Processing file 17 of 1827
0.9852216748768473% : Processing file 18 of 1827
1.0399562123700055% : Processing file 19 of 1827
1.0946907498631637% : Processing file 20 of 1827
1.1494252873563218% : Pro

## Process Loads

In [216]:
!unzip ../raw/load/Native_Load_2020.zip
!unzip ../raw/load/Native_Load_2021.zip
!unzip ../raw/load/Native_Load_2022.zip
!unzip ../raw/load/Native_Load_2023.zip
!unzip ../raw/load/Native_Load_2024.zip

Archive:  ../raw/load/Native_Load_2020.zip
  inflating: Native_Load_2020.xlsx   
Archive:  ../raw/load/Native_Load_2021.zip
  inflating: Native_Load_2021.xlsx   
Archive:  ../raw/load/Native_Load_2022.zip
  inflating: Native_Load_2022.xlsx   
Archive:  ../raw/load/Native_Load_2023.zip
  inflating: Native_Load_2023.xlsx   
Archive:  ../raw/load/Native_Load_2024.zip
  inflating: Native_Load_2024.xlsx   


In [217]:
years = ['2020', '2021', '2022', '2023', '2024']

load_df = pd.DataFrame(columns=[])
for year in years:
    df = pd.read_excel(f'./Native_Load_{year}.xlsx')
    # 2020 in particular has the column name 'HourEnding' whereas the other years have 'Hour Ending'
    df = df.rename(columns={'HourEnding': 'Hour Ending'})
    columns_to_keep = ['Hour Ending', 'WEST']
    # Drop all columns except those above
    df = df.drop(df.columns.difference(columns_to_keep), axis=1)
    load_df = pd.concat([load_df, df], ignore_index=True)
load_df.to_csv('./loads_2020-24.csv')
print(load_df)

            Hour Ending         WEST
0      01/01/2020 01:00  1172.943179
1      01/01/2020 02:00  1165.951313
2      01/01/2020 03:00  1149.076769
3      01/01/2020 04:00  1141.301918
4      01/01/2020 05:00  1147.094161
...                 ...          ...
43843  12/31/2024 20:00  1220.123968
43844  12/31/2024 21:00  1224.000675
43845  12/31/2024 22:00  1216.576523
43846  12/31/2024 23:00  1216.686183
43847  12/31/2024 24:00  1209.037402

[43848 rows x 2 columns]


## Process Prices

In [218]:
!unzip ../raw/prices/rpt.00013061.0000000000000000.20210101.084127415.RTMLZHBSPP_2020.zip
!unzip ../raw/prices/rpt.00013061.0000000000000000.20220101.083820464.RTMLZHBSPP_2021.zip
!unzip ../raw/prices/rpt.00013061.0000000000000000.20230101.083407763.RTMLZHBSPP_2022.zip
!unzip ../raw/prices/rpt.00013061.0000000000000000.20240101.081911295.RTMLZHBSPP_2023.zip
!unzip ../raw/prices/rpt.00013061.0000000000000000.20250101.081829672.RTMLZHBSPP_2024.zip

Archive:  ../raw/prices/rpt.00013061.0000000000000000.20210101.084127415.RTMLZHBSPP_2020.zip
  inflating: rpt.00013061.0000000000000000.RTMLZHBSPP_2020.xlsx  
Archive:  ../raw/prices/rpt.00013061.0000000000000000.20220101.083820464.RTMLZHBSPP_2021.zip
  inflating: rpt.00013061.0000000000000000.RTMLZHBSPP_2021.xlsx  
Archive:  ../raw/prices/rpt.00013061.0000000000000000.20230101.083407763.RTMLZHBSPP_2022.zip
  inflating: rpt.00013061.0000000000000000.RTMLZHBSPP_2022.xlsx  
Archive:  ../raw/prices/rpt.00013061.0000000000000000.20240101.081911295.RTMLZHBSPP_2023.zip
  inflating: rpt.00013061.0000000000000000.RTMLZHBSPP_2023.xlsx  
Archive:  ../raw/prices/rpt.00013061.0000000000000000.20250101.081829672.RTMLZHBSPP_2024.zip
  inflating: rpt.00013061.0000000000000000.RTMLZHBSPP_2024.xlsx  


In [219]:
prices_20 = pd.DataFrame()
prices_21 = pd.DataFrame()
prices_22 = pd.DataFrame()
prices_23 = pd.DataFrame()
prices_24 = pd.DataFrame()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for month in months:
    print(f"Processing {month}")
    prices_20 = pd.concat([prices_20, pd.read_excel(f"rpt.00013061.0000000000000000.RTMLZHBSPP_2020.xlsx", sheet_name=month)])
    prices_21 = pd.concat([prices_21, pd.read_excel(f"rpt.00013061.0000000000000000.RTMLZHBSPP_2021.xlsx", sheet_name=month)])
    prices_22 = pd.concat([prices_22, pd.read_excel(f"rpt.00013061.0000000000000000.RTMLZHBSPP_2022.xlsx", sheet_name=month)])
    prices_23 = pd.concat([prices_23, pd.read_excel(f"rpt.00013061.0000000000000000.RTMLZHBSPP_2023.xlsx", sheet_name=month)])
    prices_24 = pd.concat([prices_24, pd.read_excel(f"rpt.00013061.0000000000000000.RTMLZHBSPP_2024.xlsx", sheet_name=month)])

Processing Jan
Processing Feb
Processing Mar
Processing Apr
Processing May
Processing Jun
Processing Jul
Processing Aug
Processing Sep
Processing Oct
Processing Nov
Processing Dec


In [220]:
all_prices = pd.concat([prices_20, prices_21, prices_22, prices_23, prices_24], ignore_index=True)

# only want prices for LZ_WEST
all_prices = all_prices[all_prices['Settlement Point Type'] == 'LZ'].reset_index(drop=True)
all_prices = all_prices[all_prices['Settlement Point Name'] == 'LZ_WEST'].reset_index(drop=True)
columns_to_keep = ['Delivery Date', 'Delivery Hour', 'Delivery Interval', 'Settlement Point Price']
all_prices = all_prices.drop(all_prices.columns.difference(columns_to_keep), axis=1)

# We want to take the average price across all 4 delivery intervals, which each occur within a given hour
averaged_df = pd.DataFrame(columns=['date', 'price'])
unique_dates = all_prices['Delivery Date'].unique()
num_dates = len(unique_dates)
i = 0
for date in unique_dates:
    print(f"{(i / num_dates) * 100}% : processing price for date {i} out of {num_dates}")
    date_idxs = all_prices['Delivery Date'].isin([date])
    date_prices = all_prices[date_idxs]
    i += 1
    for hour in date_prices['Delivery Hour'].unique():
        hour_idxs = date_prices['Delivery Hour'].isin([hour])
        hour_prices = date_prices[hour_idxs]
        avg_price = hour_prices['Settlement Point Price'].mean()
        new_row = pd.DataFrame({'date':[date + ' ' + str(hour) + ':00:00'], 'price':[avg_price]})
        averaged_df = pd.concat([averaged_df, new_row], ignore_index=True)

averaged_df.to_csv('./prices_2020-24.csv')
print(averaged_df)

0.0% : processing price for date 0 out of 1827
0.05473453749315819% : processing price for date 1 out of 1827
0.10946907498631638% : processing price for date 2 out of 1827
0.16420361247947454% : processing price for date 3 out of 1827
0.21893814997263275% : processing price for date 4 out of 1827
0.27367268746579093% : processing price for date 5 out of 1827
0.3284072249589491% : processing price for date 6 out of 1827
0.38314176245210724% : processing price for date 7 out of 1827
0.4378762999452655% : processing price for date 8 out of 1827
0.49261083743842365% : processing price for date 9 out of 1827
0.5473453749315819% : processing price for date 10 out of 1827
0.60207991242474% : processing price for date 11 out of 1827
0.6568144499178982% : processing price for date 12 out of 1827
0.7115489874110563% : processing price for date 13 out of 1827
0.7662835249042145% : processing price for date 14 out of 1827
0.8210180623973727% : processing price for date 15 out of 1827
0.8757525998

/var/folders/rj/tjcj95tx32q97ft46gzv_wym0000gn/T/ipykernel_32877/3977123101.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  averaged_df = pd.concat([averaged_df, new_row], ignore_index=True)


1.5873015873015872% : processing price for date 29 out of 1827
1.6420361247947455% : processing price for date 30 out of 1827
1.6967706622879035% : processing price for date 31 out of 1827
1.751505199781062% : processing price for date 32 out of 1827
1.8062397372742198% : processing price for date 33 out of 1827
1.8609742747673783% : processing price for date 34 out of 1827
1.9157088122605364% : processing price for date 35 out of 1827
1.9704433497536946% : processing price for date 36 out of 1827
2.0251778872468527% : processing price for date 37 out of 1827
2.079912424740011% : processing price for date 38 out of 1827
2.134646962233169% : processing price for date 39 out of 1827
2.1893814997263275% : processing price for date 40 out of 1827
2.2441160372194857% : processing price for date 41 out of 1827
2.2988505747126435% : processing price for date 42 out of 1827
2.353585112205802% : processing price for date 43 out of 1827
2.40831964969896% : processing price for date 44 out of 182

# Merge all processed datasets

In [4]:
def reformatLoadTime(time):
    # Ignoring nonstandard, one-time date format mismatch
    if(time == '2022-12-01 01:00:00'):
        return time
    # Fixing nonstandard DST time format
    if(time[-4:] == ' DST'):
        time = time[:-4]
    return time + ':00'

def reformatePriceTime(time):
    if time[-10] == ' ':
        return time[:-10] + ' 0' + time[-9] + time[-6:]
    else:
        return time[:-11] + ' ' + time[-10:-8] + time[-6:]

def reformateDatetime(datetime):
    # Ignoring nonstandard, one-time date format mismatch
    if(datetime == '2022-12-01 01:00:00'):
        return datetime
    month = datetime[0:2]
    day = datetime[3:5]
    year = datetime[6:10]
    return year + '-' + month + '-' + day + datetime[10:]

In [23]:
loads = pd.read_csv('./loads_2020-24.csv')
prices = pd.read_csv('./prices_2020-24.csv')
speeds = pd.read_csv('./windspeeds_2020-24.csv')
powers = pd.read_csv('./power_2020-24.csv')

In [15]:
def getValue(series):
    if(len(series) == 0):
        return np.nan
    elif(len(series) == 1):
        return series.item()
    else:
        print(f'Encountered series of unexpected size: {len(series)}')
        return np.nan

def getDateWithHour(date, hour):
    return f'{date} {hour}:00:00'

In [24]:
merged = pd.DataFrame(columns=['datetime', 'speed', 'power', 'price', 'load'])

# Reformat some dataframes to ensure everything has the same datetime formatting
loads['Hour Ending'] = loads['Hour Ending'].map(reformatLoadTime).map(reformateDatetime)
prices['date'] = prices['date'].map(reformatePriceTime).map(reformateDatetime)

# Rename date column to datetime for all dataframes
loads = loads.rename(columns={'Hour Ending': 'datetime'})
prices = prices.rename(columns={'date': 'datetime'})
speeds = speeds.rename(columns={'time': 'datetime'})

# Remove duplicates
loads = loads.drop_duplicates(subset=['datetime'])

assert len(loads['datetime']) == len(loads['datetime'].unique())

num_dates = len(loads['datetime'])
i = 1
last_date = ''
for datetime in loads['datetime']:
    print(f'{(i / num_dates) * 100}% : Merging values at date {i} of {num_dates}')
    date = datetime[:10]
    hour = datetime[11:13]
    # Wind speeds do not have hour 24 of the last day, they have hour 00 of the next day
    if hour == '24':
        next_date = loads['datetime'][i][:10]
        fixed_datetime = getDateWithHour(next_date, '00')
        speed = getValue(speeds[speeds['datetime'] == fixed_datetime]['speed'])
    else:
        speed = getValue(speeds[speeds['datetime'] == datetime]['speed'])
    power = getValue(powers[powers['datetime'] == datetime]['ACTUAL_LZ_WEST'])
    price = getValue(prices[prices['datetime'] == datetime]['price'])
    load = getValue(loads[loads['datetime'] == datetime]['WEST'])

    new_row = pd.DataFrame({'datetime':[datetime], 'speed':[speed], 'power':[power], 'price':[price], 'load':[load]})
    merged = pd.concat([merged, new_row], ignore_index=True)
    i += 1
print(merged)

0.0022808658166640055% : Merging values at date 1 of 43843
0.004561731633328011% : Merging values at date 2 of 43843
0.0068425974499920165% : Merging values at date 3 of 43843
0.009123463266656022% : Merging values at date 4 of 43843
0.01140432908332003% : Merging values at date 5 of 43843
0.013685194899984033% : Merging values at date 6 of 43843
0.01596606071664804% : Merging values at date 7 of 43843
0.018246926533312044% : Merging values at date 8 of 43843
0.02052779234997605% : Merging values at date 9 of 43843
0.02280865816664006% : Merging values at date 10 of 43843
0.025089523983304062% : Merging values at date 11 of 43843
0.027370389799968066% : Merging values at date 12 of 43843
0.029651255616632077% : Merging values at date 13 of 43843
0.03193212143329608% : Merging values at date 14 of 43843
0.03421298724996008% : Merging values at date 15 of 43843
0.03649385306662409% : Merging values at date 16 of 43843
0.038774718883288095% : Merging values at date 17 of 43843
0.041055584

/var/folders/rj/tjcj95tx32q97ft46gzv_wym0000gn/T/ipykernel_52264/573260756.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat([merged, new_row], ignore_index=True)


0.13000935154984833% : Merging values at date 57 of 43843
0.13229021736651234% : Merging values at date 58 of 43843
0.13457108318317634% : Merging values at date 59 of 43843
0.13685194899984032% : Merging values at date 60 of 43843
0.13913281481650436% : Merging values at date 61 of 43843
0.14141368063316836% : Merging values at date 62 of 43843
0.14369454644983234% : Merging values at date 63 of 43843
0.14597541226649635% : Merging values at date 64 of 43843
0.14825627808316036% : Merging values at date 65 of 43843
0.15053714389982437% : Merging values at date 66 of 43843
0.15281800971648837% : Merging values at date 67 of 43843
0.15509887553315238% : Merging values at date 68 of 43843
0.1573797413498164% : Merging values at date 69 of 43843
0.1596606071664804% : Merging values at date 70 of 43843
0.1619414729831444% : Merging values at date 71 of 43843
0.1642223387998084% : Merging values at date 72 of 43843
0.16650320461647242% : Merging values at date 73 of 43843
0.1687840704331364

In [25]:
merged.to_csv('./dataset.csv')
print(merged.isna().sum())

datetime    0
speed       0
power       7
price       0
load        0
dtype: int64
